# 1단계: 공시 파일 파싱
## 코웨이 잠정실적 공시(HTML-XLS)에서 숫자 꺼내기

---
### 이 실습에서 배우는 것
- 파일 형식이 확장자와 다를 수 있다는 것
- BeautifulSoup으로 HTML 테이블 파싱
- 문자열 숫자 → 정수 변환
- pandas DataFrame 기초

## 1-1. 파일 형식 확인

> 공정공시 시스템에서 다운받은 `.xls` 파일은 실제로는 HTML 형식입니다.
> 파이썬에서 직접 `pd.read_excel()`로 열면 오류가 납니다.

In [ ]:
# 파일 첫 100바이트를 읽어서 실제 형식 확인
file_path = 'data/sample_disclosure.xls'  # 실제 파일 경로로 변경

with open(file_path, 'rb') as f:
    header = f.read(100)

print('파일 시작 바이트:')
print(header)

# b'<html>' 로 시작하면 HTML 파일!
if header.startswith(b'<html') or header.startswith(b'<HTML'):
    print('\n→ 이 파일은 HTML 형식입니다. BeautifulSoup으로 파싱해야 합니다.')
else:
    print('\n→ 진짜 Excel 파일입니다. pd.read_excel()로 읽으면 됩니다.')

## 1-2. BeautifulSoup으로 HTML 파싱

In [ ]:
# 필요한 라이브러리 설치 (처음 한 번만)
# !pip install beautifulsoup4 pandas

from bs4 import BeautifulSoup
import pandas as pd

# 파일 읽기 (UTF-8 인코딩)
with open(file_path, 'r', encoding='utf-8') as f:
    html_content = f.read()

# HTML 파싱
soup = BeautifulSoup(html_content, 'html.parser')

# 제목 확인
print('문서 제목:', soup.title.text)
print('테이블 개수:', len(soup.find_all('table')))

In [ ]:
# 모든 테이블의 내용 출력
tables = soup.find_all('table')

for i, table in enumerate(tables):
    rows = table.find_all('tr')
    print(f'\n=== 테이블 {i} ({len(rows)}행) ===')
    for row in rows:
        cells = [td.get_text(strip=True) for td in row.find_all(['td', 'th'])]
        if cells:  # 빈 행 제외
            print(cells)

## 1-3. 핵심 데이터 추출

In [ ]:
def parse_number(text):
    """
    '1,275,371' 같은 문자열을 정수로 변환
    '-' 나 빈 문자열은 None으로 처리
    """
    text = text.strip()
    if text in ['-', '', 'N/A']:
        return None
    try:
        return int(text.replace(',', ''))
    except ValueError:
        return None

# 테스트
test_values = ['1,275,371', '181,628', '-', '', '4,963,566']
for v in test_values:
    print(f"'{v}' → {parse_number(v)}")

In [ ]:
# 실적 테이블(인덱스 1)에서 데이터 구조화
main_table = tables[1]
rows = main_table.find_all('tr')

# 수동으로 핵심 지표 추출
# (실제 파일에서는 행 위치가 고정되어 있음)
result = {}

for row in rows:
    cells = [td.get_text(strip=True) for td in row.find_all('td')]
    if not cells:
        continue
    
    # 지표명이 첫 번째 셀에 있는 경우
    label = cells[0]
    
    if '매출액' in label:
        result['매출액_당기'] = parse_number(cells[2]) if len(cells) > 2 else None
    elif '영업이익' in label:
        result['영업이익_당기'] = parse_number(cells[2]) if len(cells) > 2 else None
    elif '당기순이익' in label and '지배기업' not in label:
        result['순이익_당기'] = parse_number(cells[2]) if len(cells) > 2 else None

print('추출된 데이터:')
for k, v in result.items():
    print(f'  {k}: {v:,} 백만원' if v else f'  {k}: 없음')

## 1-4. 실제 실습용 데이터로 이어가기

> 여러 분기 파일을 직접 파싱하는 건 복잡하므로,
> 다음 단계부터는 이미 정리된 CSV 데이터를 활용합니다.

In [ ]:
# 정리된 데이터 불러오기 (다음 단계에서 사용)
df_annual = pd.read_csv('data/coway_annual.csv')
df_quarterly = pd.read_csv('data/coway_quarterly.csv')

print('연간 데이터:')
print(df_annual.head())
print('\n분기 데이터:')
print(df_quarterly.head())

---
## ✏️ 과제

1. `parse_number()` 함수가 `'(100,000)'` (괄호로 음수 표현) 형식도 처리하도록 수정해보세요.
2. 실적 기간 테이블(테이블 0)에서 '당기실적' 날짜 범위를 추출해보세요.
3. 추출한 숫자로 **영업이익률**을 직접 계산해보세요: `영업이익 / 매출액 × 100`